# Planejamento do Projeto: Modelo de Inadimplência e Política de Crédito (WeCogno)

## 1. Objetivo

Estimar, em tempo real, a probabilidade de inadimplência de solicitações de crédito de pessoa física e converter essa probabilidade em decisões (Ex: aprovar, rejeitar, ajustar condições, encaminhar caso à mesa avaliadora, etc.) para os contratos da `base_submissao`. A entrega é composta por um modelo preditivo e por uma política de crédito, e ambos precisam ser explicáveis para quem decide, para quem audita e para o cliente afetado.

## 2. Etapas

### Etapa 1. Ambiente, carga e análise inicial das bases
Repositório, pastas, dependências versionadas, `nbstripout` e `.gitignore` (os dados nunca vão para o GitHub). Leitura dos quatro arquivos, conferência com o dicionário (69 variáveis), tipos, chaves e integridade entre as bases (todo `id_cliente` da submissão existe na cadastral? E no histórico?). Verificar se as datas são reais ou deslocamentos relativos, pois isso define o cálculo de atrasos.


### Etapa 2. Qualidade dos dados e exploração estrutural e temporal
Tratamento de nulos (e se a ausência carrega informação), duplicatas (considerando `id_contrato`, `numero_parcela` e `versao_parcela`), valores sentinela, inconsistências entre datas e valores, e outliers, com a decisão registrada para cada caso. Exploração das tabelas brutas sem olhar para o target: se `historico_parcelas` lista apenas as parcelas pagas (comparando com `qtd_parcelas_planejadas`), distribuição das datas, volume de originação por período e tempo de observação de cada contrato.


### Etapa 3. Definição do target
Construir diversas definições de inadimplência (por exemplo, atraso acima de 30, 60 ou 90 dias em qualquer parcela, ou nas primeiras N parcelas) e compará-las numa tabela única com: taxa de inadimplentes, cobertura (contratos elegíveis), taxa por safra de originação (para identificar a janela de maturação e a censura dos contratos recentes), sobreposição entre as definições, estabilidade ao longo do tempo e sentido de negócio. A escolha não usa desempenho de modelo, porque métricas de targets diferentes não são comparáveis.
*Explicabilidade:* é a decisão mais influente do projeto e é registrada com as alternativas descartadas e o motivo. A discussão do viés de seleção também entra aqui: só contratos aprovados e liberados têm comportamento observável.



### Etapa 4. Exploração com o target e testes de hipóteses
Distribuições, relação das variáveis com o target, evolução da inadimplência no tempo e comparação entre a base histórica e a `base_submissao` para detectar mudança de perfil. Hipóteses de negócio formuladas antes dos testes (renda, idade, tipo de contrato, valor da parcela sobre a renda etc.), com tamanho de efeito, intervalos de confiança e correção para múltiplos testes.
*Explicabilidade:* separar significância estatística de relevância prática.


### Etapa 5. Engenharia de atributos e modelagem
Variáveis de comportamento passado do cliente (pontualidade, atrasos, utilização) e razões financeiras (parcela/renda, crédito/bem), todas calculadas apenas com informação disponível no momento da solicitação, para evitar vazamento temporal. Tratamento de categóricas e nulos dentro de `Pipeline` e `ColumnTransformer`, para evitar vazamento entre treino e validação. Baseline interpretável (regressão logística, com ou sem WOE), seguido de gradient boosting, com validação temporal, métricas de discriminação (AUC, KS, Gini) e de calibração (curva de calibração, Brier score), pois a probabilidade será usada diretamente na política. Tuning com critério explícito, seeds fixas e código reutilizável em `src/`.
*Explicabilidade:* preferir variáveis com sentido de negócio claro, comparar os modelos também pelo custo de interpretação e avaliar restrições de monotonicidade. Se um modelo complexo ganhar pouco sobre o interpretável, a preferência é pelo interpretável.


### Etapa 6. Explicabilidade, equidade e conformidade
Importância global (permutation importance e SHAP), efeito marginal das principais variáveis (PDP ou ALE), explicações individuais por contrato (principais fatores que elevaram ou reduziram o risco) e verificação de que os efeitos fazem sentido de negócio. Estabilidade das variáveis e do score ao longo do tempo (PSI), desempenho por segmentos e análise de impacto desigual. Avaliação do uso de `sexo` e `estado_civil` à luz de equidade e LGPD, incluindo o direito à revisão de decisões automatizadas (a validação jurídica final cabe à área responsável). Elaboração do model card com finalidade, dados, métricas, limitações e uso indevido.


### Etapa 7. Política de crédito e aplicação à `base_submissao`
Converter a probabilidade em decisão: pontos de corte, faixas de risco, ajustes de condições (valor, parcela, entrada) e encaminhamento de casos limítrofes à mesa avaliadora, com premissas de perda esperada explícitas e o trade-off entre aprovação e inadimplência. Aplicação à `base_submissao`, gerando probabilidade, faixa, decisão e fatores principais de cada solicitação, e verificando a coerência da distribuição de resultados com o histórico.
*Explicabilidade:* a política deve poder ser lida sem o modelo (regras e faixas claras), e cada decisão vem com seus motivos principais.


### Etapa 8. Monitoramento, documentação e entrega
Plano de monitoramento (estabilidade, desempenho e calibração, com gatilhos para reavaliação ou retreino), conclusões, limitações, riscos, recomendações e narrativa para o público de negócio e para a auditoria.


## 3. Acompanhamento

| Etapa | Entrega principal | Status |
|---|---|---|
| 1 | Ambiente, carga e auditoria | Em andamento |
| 2 | Qualidade e exploração estrutural | Pendente |
| 3 | Definição do target | Pendente |
| 4 | Exploração com target e hipóteses | Pendente |
| 5 | Atributos e modelagem | Pendente |
| 6 | Explicabilidade, equidade e conformidade | Pendente |
| 7 | Política de crédito e aplicação | Pendente |
| 8 | Monitoramento e entrega | Pendente |